# Remaining EEG targets: grammar, word identity, repetition, semantics, reading task

**CPU → Run all, Zyion account.** Everything runs in one notebook. No EEG re-extraction. Outputs are separate from every previous experiment.

## What this adds
| Target | Dataset | Labels and exclusions |
|---|---|---|
| Part of speech | TeCo task1 | Noun / verb / adjective from Persian Stanza. Exclude other POS and any token alignment failure. |
| Content/function | TeCo task1 | New automatic POS-derived content/function labels. Exclude punctuation, numbers and unresolved categories. These are NOT the earlier agent grammar labels. |
| Word identity | TeCo task1 | Separate5-word and10-word tasks. Vocabulary fixed by complete-source frequencies before EEG fitting: ≥3textual occurrences, ≥6readers, ≥50accepted EEG observations per word. Export the actual words. |
| Repetition | TeCo task1 | First versus later textual occurrence of a word form, per reader/task, using ALL source metadata including unfixated words. This is not refixation or lifetime familiarity. |
| Semantic category | **ZuCo SR only** | English common nouns with unambiguous WordNet sense ancestry: physical versus abstract. All mapped noun senses must agree; unknown/ambiguous words excluded. These are lexical ontology labels, not human concreteness ratings or contextual sense disambiguation. NLTK's public OMW package does NOT provide Persian, so we do not invent Persian semantic labels. |
| Reading task | **ZuCo SR + NR + TSR** | Recorded task identity. Task, sentence material and recording session can be confounded; success is not proof of a pure task-related brain state. |

**No word-length classification.** Word length/position, sentence length and control status appear only in the metadata-only comparator. Text, word IDs, POS labels, reader IDs and task IDs never enter EEG feature vectors.

## Setups
- Reuse15regional/time/amplitude representations from the existing48regional voltage features; pre-onset and metadata-only controls are separate. No high-resolution source signals are opened.
- Exact individual trials, exact10-trial averages, and **up-to40** averages including remainders. All average within reader/target/partition. Known target labels construct class averages; these scores are not individual-word deployment accuracies.
- Familiar identities, unseen sentences, unseen readers; **unseen word identities** also for POS/content-function/semantic/repetition tasks. Word-identity classification cannot test an unseen word as a known target class.
- Familiar means every retained test reader, sentence and word identity is in training AFTER overlap purging. Every test EEG event remains separate. Purge training epochs within1000ms of test/validation events in the same reader/task/sentence, also in inner validation. Do not mix identically numbered sentence positions from different ZuCo tasks.
-3outerfolds/2innerfolds. Screen all EEG inputs with C1logistic; best3advance to nine model settings: logistic C=.1/1/10, shrinkageLDA, approximateRBF, ExtraTrees, RandomForest, small32-unit MLP and kNN. The held-out test fold never chooses the winner. Display **fixed EEG and selected EEG separately**, plus pre-onset, metadata-only and dummy.
- To bound runtime on combined ZuCo, **every fit uses at most10,000training examples**, deterministically stratified before any preprocessing. Models share the same subsample. **All held-out examples remain in evaluation**. Store the selection digest; original memberships plus seed reproduce it. Unavailable classes/groups are reported without changing the rules.

## Automatic annotation and privacy
Stanza runs on the complete native source word sequence, with pretokenized original word indices. Never annotate only the words the participant fixated. Save exact word alignment and exclude failures. WordNet labels use only English noun senses for the separate ZuCo semantic experiment. Every automatic label, source word and exclusion reason is saved privately as inspectable CSV/JSON.

First run downloads **public Stanza models, English WordNet and CPU dependencies to temporary Colab disk**, estimated **1–2GB total temporary footprint**; no Mac installation and no model copy on Drive. The models run inside Colab; your text is not submitted to an online labeling API. Complete per-language label files resume without downloading models or relabeling again. Model artifacts use the published Stanza MD5 checks.

**Drive cap:50MB** under `Thesis/Results/word_sentiment_eeg/linguistic_targets/`, including small label tables and compressed fit checkpoints. No new EEG/feature copies, no saved model bank, no deletion. Allow several hours; this larger run is not benchmarked. Logs show cached reader X/N, annotation sentence X/N and target/split/fold/setup. No GPU required.

This is a fixed exploratory plan, not a promise of high accuracy, an exact paper reproduction or a significance search. NLP labels are automatically generated, not human consensus. Inspect label coverage and metadata/pre-onset controls when interpreting results.


In [ ]:
# Runtime check: run before installation, Drive access, or analysis.
import os
import subprocess

REQUIRED_RUNTIME = 'CPU'
if os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME'):
    raise RuntimeError(f'This notebook needs {REQUIRED_RUNTIME}, not TPU. Choose Runtime > Change runtime type, then rerun all.')
try:
    _gpu_probe = subprocess.run(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15, check=False)
except FileNotFoundError:
    _gpu_names = ''
except subprocess.TimeoutExpired:
    raise RuntimeError('GPU detection timed out. Reconnect the runtime and rerun this cell.') from None
else:
    if _gpu_probe.returncode:
        raise RuntimeError('GPU detection failed. Reconnect the runtime and rerun this cell.')
    _gpu_names = _gpu_probe.stdout.strip()
if REQUIRED_RUNTIME == 'GPU' and not _gpu_names:
    raise RuntimeError('GPU required. Choose Runtime > Change runtime type > T4 GPU, then Run all. No setup or analysis has started.')
if REQUIRED_RUNTIME == 'CPU' and _gpu_names:
    raise RuntimeError('CPU notebook: choose Runtime > Change runtime type > CPU (or hardware accelerator None), then Run all to avoid using GPU quota.')
print('Runtime verified:', _gpu_names if _gpu_names else 'CPU', flush=True)


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/linguistic_targets'
REPO=Path('/content/word-sentiment-eeg-linguistic-targets')
CODE_REVISION='PIN_AFTER_TEST'
def git(*args):
    result=subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],
                          text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.returncode:
        raise RuntimeError('Git setup failed:\n'+result.stdout)
    if result.stdout:print(result.stdout,end='',flush=True)
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code differs from the pinned commit.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    raise RuntimeError('Code checkout verification failed.')
# Temporary cloud-only scientific environment: no base-runtime upgrades/restart.
# BOOTSTRAP_START: also exercised verbatim in cloud CI.
# Use a file-rendering backend in subprocesses, not Colab's unavailable inline plugin.
os.environ['MPLBACKEND']='Agg'
ENV=Path('/content/word-eeg-linguistic-python')
PYTHON=ENV/'bin/python'
def setup_command(command):
    print('SETUP:', ' '.join(map(str,command)), flush=True)
    result=subprocess.run(list(map(str,command)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    if result.stdout:print(result.stdout,flush=True)
    if result.returncode:
        raise RuntimeError('Scientific environment setup failed; the full diagnostic is printed above. No EEG processing started.')
# Colab may lack ensurepip. This also repairs a partially created environment.
setup_command([sys.executable,'-m','venv','--without-pip',str(ENV)])
# CPU wheels and NLP packages are downloaded ONLY in Colab. No Mac installation.
setup_command([sys.executable,'-m','pip','--python',str(PYTHON),'install',
               'torch==2.8.0','--index-url','https://download.pytorch.org/whl/cpu'])
setup_command([sys.executable,'-m','pip','--python',str(PYTHON),'install',
               '--only-binary=:all:','-r',str(REPO/'requirements-linguistic.txt')])
setup_command([sys.executable,'-m','pip','--python',str(PYTHON),'check'])
setup_command([str(PYTHON),'-c',
    'import numpy,pandas,scipy,sklearn,matplotlib; '
    'print("Scientific environment ready:",numpy.__version__,pandas.__version__,scipy.__version__,sklearn.__version__,matplotlib.__version__)'])
# BOOTSTRAP_END
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# Verify account quota before loading any EEG. Use Zyion for both prompts.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();about=response.json();quota=about['storageQuota']
email=about.get('user',{}).get('emailAddress','unavailable')
print('Authenticated quota account:',email)
if email!='zyionygvmutch39@gmail.com':
    raise RuntimeError('Choose your Zyion account for Drive authentication and mounting; it contains the saved trials.')
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a storage limit; cannot verify free space.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
used=sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) if OUT.exists() else 0
if FREE_DRIVE_BYTES<max(0,50_000_000-used):
    raise RuntimeError('Reserve at most 50 MB for results. This notebook reads existing feature caches only; no EEG shards or extraction.')
print(f'Free space: {FREE_DRIVE_BYTES/1e9:.2f} GB; new output cap: 50 MB.')
print('New output folder:',OUT)


In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[str(PYTHON),'-u',str(REPO/'scripts/linguistic_targets.py'),'--thesis-root',str(THESIS),'--free-drive-bytes',str(FREE_DRIVE_BYTES)]
run_with_progress(command,OUT/'runtime_status.json',activity='Remaining linguistic EEG targets: cache-only data and automatic labels')


In [ ]:
import hashlib, gzip
from IPython.display import display, Markdown, Image
state=json.loads((OUT/'complete.json').read_text())
assert state['status']=='completed'
for name,sha in state['checksums'].items():
    if hashlib.sha256((OUT/name).read_bytes()).hexdigest()!=sha:raise RuntimeError('Checksum failed: '+name)
summary=pd.read_csv(OUT/'summary.csv',dtype={'size':str})
display(Markdown('## Finished and verified — all held-out test results'))
for target,g in summary.groupby('target',sort=False):
    display(Markdown('### '+target))
    with pd.option_context('display.max_rows',200,'display.max_columns',20):display(g.round(3))
display(Image(filename=str(OUT/'overview.png')))
if (OUT/'reader_confusion.png').is_file():display(Image(filename=str(OUT/'reader_confusion.png')))


In [ ]:
display(Markdown('## Target definitions and sample counts'))
for path in sorted((OUT/'targets').glob('*/definition.json')):
    value=json.loads(path.read_text());print(path.parent.name, json.dumps(value,ensure_ascii=False))
display(Markdown('## Chosen configurations — compare selected_eeg with fixed_eeg'))
rows=json.loads(gzip.decompress((OUT/'outer_results.json.gz').read_bytes()))
chosen=pd.DataFrame([{k:r.get(k) for k in ['target','mode','fold','size','variant','input','model','status','balanced_accuracy','test_examples','actual_group_min','actual_group_max','reason']} for r in rows])
with pd.option_context('display.max_rows',300):display(chosen[chosen.variant.isin(['selected_eeg','fixed_eeg'])].round(3))
display(Markdown('## Unavailable conditions'))
missing=pd.read_csv(OUT/'unavailable.csv')
with pd.option_context('display.max_rows',200):display(missing)
print('Saved results:', OUT)
print('summary.csv = final scores; development.csv.gz = validation search; outer_results.json.gz = predictions; fits/ = resumable compressed checkpoints and exact event memberships.')

print('Private annotation files:')
for path in sorted((OUT/'labels').glob('*/words.csv.gz')):
    frame=pd.read_csv(path,keep_default_na=False)
    print(path, 'words:',len(frame))
    display(frame.head(20))
    print('POS counts:',frame.upos.value_counts().to_dict())
    print('Semantic coverage:',frame.semantic_class.value_counts().to_dict())
